In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split # <-- ADDED THIS

# ==========================================
# 1. SETUP & LOAD BRFSS 2019
# ==========================================
print("Setting up BRFSS 2019 dataset...")
base_dir = '/content/drive/MyDrive/Arthritis version 2/BRFSS 2019/'
os.makedirs(base_dir, exist_ok=True)

file_path_2019 = os.path.join(base_dir, 'BRFSS_2019.csv')

try:
    df_brfss_2019 = pd.read_csv(file_path_2019)
    print(f"Original 2019 shape loaded: {df_brfss_2019.shape}")
except FileNotFoundError:
    print(f"⚠️ Error: Could not find the 2019 dataset at {file_path_2019}. Please check the filename.")

# ==========================================
# 2. APPLY EXACT 2021 MAPPINGS & BINNINGS
# ==========================================
print("Applying strict 2021 codebook mappings to 2019 data...")

df_brfss_2019.columns = df_brfss_2019.columns.str.upper()

rename_dict = {
    'ACEDIVRC': 'Parents Separation', 'ACEPUNCH': 'Physically hurt', 'ACEDEPRS': 'Depressed Acompany',
    'ACESWEAR': 'Verbally Insulted', 'ACETOUCH': 'Touched Sexually', 'ACEHVSEX': 'Sexual harassment',
    'ACEDRUGS': 'Drug Addicted F_member', '_AGEG5YR': 'Age', '_RACE': 'Race', '_SEX': 'Gender',
    'INCOME2': 'Income Level', 'INCOME3': 'Income Level', 'EDUCA': 'Education level', 'EMPLOY1': 'Employment statues',
    'MARITAL': 'Marital status', 'SMOKE100': 'Smoking status', 'ALCDAY5': 'Alcohol consumption',
    'EXERANY2': 'Physical activity', 'CHECKUP1': 'Regular Checkup', 'FRUIT2': 'Dietary Habits (Fruits)',
    'VEGETAB2': 'Dietary Habits (Vegetables)', 'RENTHOM1': 'Housing Status', 'DEAF': 'DEAF',
    '_BMI5CAT': 'BMI', 'BLIND': 'BLIND', 'DECIDE': 'Difficulty concentrating', 'DIFFWALK': 'DIFFWALK',
    'HLTHPLN1': 'Health Insurance', '_HLTHPLN': 'Health Insurance', '_MENT14D': 'Mental Health',
    '_PHYS14D': 'Physical Health', '_RFHYPE5': 'Blood pressure', '_RFHYPE6': 'Blood pressure',
    '_CHOLCH2': 'Cholesterol Awareness', '_CHOLCH3': 'Cholesterol Awareness',
    'HAVARTH4': 'Arthritis', 'HAVARTH5': 'Arthritis'
}

df_brfss_2019 = df_brfss_2019.rename(columns=rename_dict)
available_features = [col for col in rename_dict.values() if col in df_brfss_2019.columns]
available_features = list(dict.fromkeys(available_features))
df_clean = df_brfss_2019[available_features].copy()

# Custom Values Replacement
df_clean['Parents Separation'] = df_clean['Parents Separation'].replace({2.0: 0, 8.0: 2, 7.0: np.nan, 9.0: np.nan})
df_clean['Physically hurt'] = df_clean['Physically hurt'].replace({1.0: 0, 2.0: 1, 3.0: 2, 7.0: np.nan, 9.0: np.nan})
df_clean['Depressed Acompany'] = df_clean['Depressed Acompany'].replace({2.0: 0, 7.0: np.nan, 9.0: np.nan})
df_clean['Verbally Insulted'] = df_clean['Verbally Insulted'].replace({1.0: 0, 2.0: 1, 3.0: 2, 7.0: np.nan, 9.0: np.nan})
df_clean['Touched Sexually'] = df_clean['Touched Sexually'].replace({1.0: 0, 2.0: 1, 3.0: 2, 7.0: np.nan, 9.0: np.nan})
df_clean['Sexual harassment'] = df_clean['Sexual harassment'].replace({1.0: 0, 2.0: 1, 3.0: 2, 7.0: np.nan, 9.0: np.nan})
df_clean['Drug Addicted F_member'] = df_clean['Drug Addicted F_member'].replace({2.0: 0, 7.0: np.nan, 9.0: np.nan})
df_clean['Age'] = df_clean['Age'].replace({14.0: np.nan})
df_clean['Race'] = df_clean['Race'].replace({9.0: np.nan})
df_clean['Gender'] = df_clean['Gender'].replace({2.0: 0, 7.0: np.nan, 8.0: np.nan, 9.0: np.nan})
df_clean['Income Level'] = df_clean['Income Level'].replace({77.0: np.nan, 99.0: np.nan})
df_clean['Education level'] = df_clean['Education level'].replace({1.0: 0, 2.0: 1, 3.0: 2, 4.0: 3, 5.0: 4, 6.0: 5, 9.0: np.nan})
df_clean['Employment statues'] = df_clean['Employment statues'].replace({9.0: np.nan})
df_clean['Marital status'] = df_clean['Marital status'].replace({5.0: 0, 6.0: 5, 9.0: np.nan})
df_clean['Smoking status'] = df_clean['Smoking status'].replace({2.0: 0, 7.0: np.nan, 9.0: np.nan})

alc_replace = {
    888.0: 0, 201: 1, 202: 2, 203: 3, 204: 4, 101: 4, 205: 5, 206: 6, 207: 7, 208: 8, 102: 8, 209: 9,
    210: 10, 211: 11, 212: 12, 103: 12, 213: 13, 214: 14, 215: 15, 216: 16, 104: 16, 217: 17, 218: 18,
    219: 19, 220: 20, 105: 20, 221: 21, 222: 22, 223: 23, 224: 24, 106: 24, 225: 25, 226: 26, 227: 27,
    228: 28, 107: 28, 229: 29, 230: 30, 777.0: np.nan, 999.0: np.nan
}
df_clean['Alcohol consumption'] = df_clean['Alcohol consumption'].replace(alc_replace)

df_clean['Physical activity'] = df_clean['Physical activity'].replace({2.0: 0, 7.0: np.nan, 9.0: np.nan})
df_clean['Regular Checkup'] = df_clean['Regular Checkup'].replace({8.0: 0, 7.0: np.nan, 9.0: np.nan})

diet_fruit = {555.0: 0, 300.0: 0, 777.0: np.nan, 999.0: np.nan}
for val in [101.0, 201.0, 301.0]: diet_fruit[val] = 1
for val in list(range(102, 199)) + list(range(202, 299)) + list(range(302, 399)): diet_fruit[float(val)] = 2
for val in [199.0, 299.0, 399.0]: diet_fruit[val] = 3
df_clean['Dietary Habits (Fruits)'] = df_clean['Dietary Habits (Fruits)'].replace(diet_fruit)

diet_veg = {555.0: 0, 300.0: 0, 777.0: np.nan, 999.0: np.nan}
for val in [101.0, 201.0, 301.0]: diet_veg[val] = 1
for val in list(range(102, 199)) + list(range(202, 299)) + list(range(302, 399)): diet_veg[float(val)] = 2
for val in [199.0, 299.0, 399.0]: diet_veg[val] = 3
df_clean['Dietary Habits (Vegetables)'] = df_clean['Dietary Habits (Vegetables)'].replace(diet_veg)

df_clean['Housing Status'] = df_clean['Housing Status'].replace({7.0: np.nan, 9.0: np.nan})
df_clean['DEAF'] = df_clean['DEAF'].replace({2.0: 0, 7.0: np.nan, 9.0: np.nan})
df_clean['BLIND'] = df_clean['BLIND'].replace({2.0: 0, 7.0: np.nan, 9.0: np.nan})
df_clean['Difficulty concentrating'] = df_clean['Difficulty concentrating'].replace({2.0: 0, 7.0: np.nan, 9.0: np.nan})
df_clean['DIFFWALK'] = df_clean['DIFFWALK'].replace({2.0: 0, 7.0: np.nan, 9.0: np.nan})
df_clean['Health Insurance'] = df_clean['Health Insurance'].replace({2.0: 0, 7.0: np.nan, 9.0: np.nan})
df_clean['Mental Health'] = df_clean['Mental Health'].replace({1.0: 0, 2.0: 1, 3.0: 2, 9.0: np.nan})
df_clean['Physical Health'] = df_clean['Physical Health'].replace({1.0: 0, 2.0: 1, 3.0: 2, 9.0: np.nan})
df_clean['Blood pressure'] = df_clean['Blood pressure'].replace({2.0: 0, 7.0: np.nan, 9.0: np.nan})
df_clean['Cholesterol Awareness'] = df_clean['Cholesterol Awareness'].replace({1.0: 0, 2.0: 1, 3.0: 2, 7.0: np.nan, 9.0: np.nan})
df_clean['Arthritis'] = df_clean['Arthritis'].replace({2.0: 0, 7.0: np.nan, 9.0: np.nan})

# ==========================================
# 3. CLEAN TARGET AND ISOLATE 20%
# ==========================================
print("\nDropping rows missing the target variable (Arthritis)...")
df_clean = df_clean.dropna(subset=['Arthritis']).reset_index(drop=True)

print("Splitting dataset to isolate exactly 20% (Stratified) for testing...")
# Using train_test_split to strictly preserve the class balance (stratify)
_, df_test = train_test_split(df_clean, test_size=0.2, stratify=df_clean['Arthritis'], random_state=42)
df_test = df_test.reset_index(drop=True)

print(f"Sampled 20% data shape ready for GAN Imputation: {df_test.shape}")

# ==========================================
# 4. PREPARE FOR GAN NETWORK
# ==========================================
# Use df_test instead of df_clean!
X_test_2019 = df_test.drop('Arthritis', axis=1)
y_test_2019 = df_test['Arthritis']

nan_mask = X_test_2019.isna()
input_dim = X_test_2019.shape[1]

imputer = SimpleImputer(strategy='mean')
X_temp = imputer.fit_transform(X_test_2019)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_temp)

feature_limits = {
    'Physically hurt': (0, 2), 'Depressed Acompany': (0, 1), 'Verbally Insulted': (0, 2),
    'Touched Sexually': (0, 2), 'Sexual harassment': (0, 2), 'Drug Addicted F_member': (0, 1),
    'Age': (1, 13), 'Gender': (0, 1), 'Income Level': (1, 11), 'Education level': (0, 5),
    'Smoking status': (0, 1), 'Alcohol consumption': (0, 30), 'Physical activity': (0, 1),
    'Regular Checkup': (0, 4), 'Dietary Habits (Fruits)': (0, 3), 'Dietary Habits (Vegetables)': (0, 3),
    'DEAF': (0, 1), 'BMI': (1, 4), 'BLIND': (0, 1), 'Difficulty concentrating': (0, 1),
    'DIFFWALK': (0, 1), 'Health Insurance': (0, 1), 'Mental Health': (0, 2),
    'Physical Health': (0, 2), 'Blood pressure': (0, 1)
}

def finalize_imputation(X_original, X_reconstructed):
    X_recon_inv = scaler.inverse_transform(X_reconstructed)
    X_recon_df = pd.DataFrame(X_recon_inv, columns=X_original.columns)
    X_final = X_original.copy()

    for col in X_final.columns:
        X_final.loc[nan_mask[col], col] = X_recon_df.loc[nan_mask[col], col]

    X_final = X_final.round().astype(int)

    for feature, (lower, upper) in feature_limits.items():
        if feature in X_final.columns:
            X_final[feature] = X_final[feature].clip(lower=lower, upper=upper)

    return X_final

BATCH_SIZE = 256

# ==========================================
# 5. GAN IMPUTATION
# ==========================================
print("\nTraining GAN Imputer on 20% BRFSS 2019 sample... (This will be much faster)")
generator = Sequential([
    Input(shape=(input_dim,)),
    Dense(128, activation='relu'),
    Dense(256, activation='relu'),
    Dense(input_dim, activation='sigmoid')
])
discriminator = Sequential([
    Input(shape=(input_dim,)),
    Dense(256, activation='relu'),
    Dropout(0.2),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])
discriminator.compile(optimizer=Adam(learning_rate=0.0002), loss='binary_crossentropy')
discriminator.trainable = False

gan_input = Input(shape=(input_dim,))
generated_data = generator(gan_input)
gan_output = discriminator(generated_data)
gan = Model(gan_input, gan_output)
gan.compile(optimizer=Adam(learning_rate=0.0002), loss='binary_crossentropy')

batch_count = int(X_scaled.shape[0] / BATCH_SIZE)
GAN_EPOCHS = 100
for epoch in range(GAN_EPOCHS):
    for _ in range(batch_count):
        real_data = X_scaled[np.random.randint(0, X_scaled.shape[0], size=BATCH_SIZE)]
        noise = np.random.normal(0, 1, size=[BATCH_SIZE, input_dim])
        fake_data = generator.predict(real_data + noise * 0.1, verbose=0)

        discriminator.train_on_batch(real_data, np.ones((BATCH_SIZE, 1)))
        discriminator.train_on_batch(fake_data, np.zeros((BATCH_SIZE, 1)))

        noise = np.random.normal(0, 1, size=[BATCH_SIZE, input_dim])
        gan.train_on_batch(real_data + noise * 0.1, np.ones((BATCH_SIZE, 1)))

print("Reconstructing missing values with GAN...")
X_gan_pred = generator.predict(X_scaled, verbose=0)
X_gan_final = finalize_imputation(X_test_2019, X_gan_pred)

# ==========================================
# 6. ONE-HOT ENCODING (ALIGN WITH 2021 MODELS)
# ==========================================
print("\nAligning features with 2021 training data (One-Hot Encoding)...")

categorical_cols = [
    'Parents Separation', 'Race', 'Employment statues',
    'Marital status', 'Housing Status', 'Cholesterol Awareness'
]

expected_features = [
    'Physically hurt', 'Depressed Acompany', 'Verbally Insulted', 'Touched Sexually',
    'Sexual harassment', 'Drug Addicted F_member', 'Age', 'Gender', 'Income Level',
    'Education level', 'Smoking status', 'Alcohol consumption', 'Physical activity',
    'Regular Checkup', 'Dietary Habits (Fruits)', 'Dietary Habits (Vegetables)',
    'DEAF', 'BMI', 'BLIND', 'Difficulty concentrating', 'DIFFWALK', 'Health Insurance',
    'Mental Health', 'Physical Health', 'Blood pressure',
    'Parents Separation_0.0', 'Parents Separation_1.0', 'Parents Separation_2.0', 'Parents Separation_nan',
    'Race_1.0', 'Race_2.0', 'Race_3.0', 'Race_4.0', 'Race_5.0', 'Race_6.0', 'Race_7.0', 'Race_8.0', 'Race_nan',
    'Employment statues_1.0', 'Employment statues_2.0', 'Employment statues_3.0', 'Employment statues_4.0',
    'Employment statues_5.0', 'Employment statues_6.0', 'Employment statues_7.0', 'Employment statues_8.0', 'Employment statues_nan',
    'Marital status_0.0', 'Marital status_1.0', 'Marital status_2.0', 'Marital status_3.0', 'Marital status_4.0',
    'Marital status_5.0', 'Marital status_nan',
    'Housing Status_1.0', 'Housing Status_2.0', 'Housing Status_3.0', 'Housing Status_nan',
    'Cholesterol Awareness_0.0', 'Cholesterol Awareness_1.0', 'Cholesterol Awareness_2.0', 'Cholesterol Awareness_nan'
]

for col in categorical_cols:
    X_gan_final[col] = X_gan_final[col].astype(float)

X_encoded = pd.get_dummies(X_gan_final, columns=categorical_cols, dummy_na=True, dtype=int)
X_aligned = X_encoded.reindex(columns=expected_features, fill_value=0)

# ==========================================
# 7. SAVE FINAL DATASET
# ==========================================
df_final_test = pd.concat([X_aligned, y_test_2019], axis=1)

# Saving with a new name reflecting the 20% sample
out_path = os.path.join(base_dir, 'BRFSS_2019_Test_20Percent_GAN.csv')
df_final_test.to_csv(out_path, index=False)

print(f"✅ Saved 20% Stratified, Imputed & Encoded 2019 Test Set ({len(df_final_test):,} rows, {df_final_test.shape[1]} columns)")
print(f"File saved as: '{out_path}'")
print("\n🎉 BRFSS DATA PREP FINISHED SUCCESSFULLY!")

Setting up BRFSS 2019 dataset...
Original 2019 shape loaded: (418268, 342)
Applying strict 2021 codebook mappings to 2019 data...

Dropping rows missing the target variable (Arthritis)...
Splitting dataset to isolate exactly 20% (Stratified) for testing...
Sampled 20% data shape ready for GAN Imputation: (83145, 32)

Training GAN Imputer on 20% BRFSS 2019 sample... (This will be much faster)
Reconstructing missing values with GAN...

Aligning features with 2021 training data (One-Hot Encoding)...
✅ Saved 20% Stratified, Imputed & Encoded 2019 Test Set (83,145 rows, 63 columns)
File saved as: '/content/drive/MyDrive/Arthritis version 2/BRFSS 2019/BRFSS_2019_Test_20Percent_GAN.csv'

🎉 BRFSS DATA PREP FINISHED SUCCESSFULLY!


In [4]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix

# ==========================================
# 1. SETUP PATHS AND LOAD 2019 DATA
# ==========================================
print("Loading BRFSS 2019 20% Stratified GAN-imputed test data...")
base_dir = '/content/drive/MyDrive/Arthritis version 2/'

# --- UPDATED PATH: Pointing exactly to the newly saved 20% GAN dataset ---
data_2019_path = os.path.join(base_dir, 'BRFSS 2019/BRFSS_2019_Test_20Percent_GAN.csv')

df_2019 = pd.read_csv(data_2019_path)
X_test_2019 = df_2019.drop('Arthritis', axis=1)
y_test_2019 = df_2019['Arthritis']

print(f"2019 Data loaded! Shape: {X_test_2019.shape}\n")

# ==========================================
# 2. DEFINE THE MODELS TO LOAD
# ==========================================
models_dir = os.path.join(base_dir, 'BRFSS 2021/Saved_Models_For_2019_Eval/')

model_files = {
    "GAN Imputation RUS Stacked Ensemble": "Stacked_Ensemble_RUS.pkl",
    "GAN Imputation ROS Stacked Ensemble": "GAN_ROS_Stacked_Ensemble.joblib",
    "GAN Imputation ROS XGBoost":          "GAN_ROS_XGBoost.joblib",
    "GAN Imputation ROS Logistic Reg":     "GAN_ROS_Logistic_Reg.joblib",
    "GAN Imputation RUS Logistic Reg":     "GAN_RUS_Logistic_Reg.joblib"
}

# ==========================================
# 3. EVALUATION FUNCTION
# ==========================================
def evaluate_model(model, X, y):
    """Calculates AUROC, Balanced Accuracy, Sensitivity, and Specificity"""
    # Get predictions and probabilities
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1] # Probabilities for the positive class (1)

    # Calculate Metrics
    auroc = roc_auc_score(y, y_proba)
    bal_acc = balanced_accuracy_score(y, y_pred)

    # Confusion Matrix for Sens/Spec
    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    # Composite Score
    composite_score = (auroc + bal_acc + sensitivity + specificity) / 4

    return {
        "Composite Score": composite_score,
        "AUROC": auroc,
        "Balanced Accuracy": bal_acc,
        "Overall Sensitivity": sensitivity,
        "Overall Specificity": specificity
    }

# ==========================================
# 4. RUN EVALUATION
# ==========================================
print("Starting evaluation on BRFSS 2019...\n")
results_list = []

for model_name, file_name in model_files.items():
    model_path = os.path.join(models_dir, file_name)

    try:
        print(f"Loading and evaluating: {model_name}...")

        # Load the model (joblib handles .pkl files perfectly)
        model = joblib.load(model_path)

        # Evaluate
        metrics = evaluate_model(model, X_test_2019, y_test_2019)

        # Add identifiers for the final table
        parts = model_name.split(" ")
        fill_method = " ".join(parts[:2]) # "GAN Imputation"
        sampling = parts[2]               # "RUS" or "ROS"
        base_model = " ".join(parts[3:])  # "Stacked Ensemble", etc.

        row = {
            "Fill method": fill_method,
            "Sampling": sampling,
            "Model": base_model,
            **metrics
        }
        results_list.append(row)

    except FileNotFoundError:
        print(f"  -> ERROR: Could not find '{file_name}' in {models_dir}. Please check the filename.")
    except Exception as e:
        print(f"  -> ERROR evaluating {model_name}: {e}")

# ==========================================
# 5. DISPLAY RESULTS
# ==========================================
print("\nEvaluation Complete! Here are the results for BRFSS 2019:\n")

# Convert to DataFrame for a clean, tabular view
results_df = pd.DataFrame(results_list)

# Sort by Composite Score descending
if not results_df.empty:
    results_df = results_df.sort_values(by="Composite Score", ascending=False).reset_index(drop=True)
    display(results_df) # Use print(results_df) if not in Jupyter/Colab

    # --- UPDATED PATH: Saving these results to a CSV with a new name reflecting the 20% sample ---
    results_out_path = os.path.join(base_dir, 'BRFSS_2019_20Percent_GAN_Evaluation_Results.csv')
    results_df.to_csv(results_out_path, index=False)
    print(f"\nResults saved to: {results_out_path}")

Loading BRFSS 2019 20% Stratified GAN-imputed test data...
2019 Data loaded! Shape: (83145, 62)

Starting evaluation on BRFSS 2019...

Loading and evaluating: GAN Imputation RUS Stacked Ensemble...
Loading and evaluating: GAN Imputation ROS Stacked Ensemble...
Loading and evaluating: GAN Imputation ROS XGBoost...
Loading and evaluating: GAN Imputation ROS Logistic Reg...
Loading and evaluating: GAN Imputation RUS Logistic Reg...

Evaluation Complete! Here are the results for BRFSS 2019:



,Fill method,Sampling,Model,Composite Score,AUROC,Balanced Accuracy,Overall Sensitivity,Overall Specificity
0,GAN Imputation,RUS,Logistic Reg,0.740993,0.797486,0.722162,0.798755,0.645569
1,GAN Imputation,ROS,Logistic Reg,0.740962,0.797623,0.722075,0.790738,0.653413
2,GAN Imputation,RUS,Stacked Ensemble,0.740956,0.800734,0.721030,0.806020,0.636041
3,GAN Imputation,ROS,Stacked Ensemble,0.729444,0.785181,0.710865,0.744569,0.677161
4,GAN Imputation,ROS,XGBoost,0.729368,0.785763,0.710570,0.751333,0.669807



Results saved to: /content/drive/MyDrive/Arthritis version 2/BRFSS_2019_20Percent_GAN_Evaluation_Results.csv


In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix

# ==========================================
# 1. SETUP PATHS AND LOAD 2019 DATA
# ==========================================
print("Loading BRFSS 2019 10% Sampled GAN-imputed test data...")
base_dir = '/content/drive/MyDrive/Arthritis version 2/'

# --- UPDATED PATH: Pointing exactly to the newly saved 10% GAN dataset ---
data_2019_path = os.path.join(base_dir, 'BRFSS 2019/BRFSS_2019_Test_10Percent_GAN.csv')

df_2019 = pd.read_csv(data_2019_path)
X_test_2019 = df_2019.drop('Arthritis', axis=1)
y_test_2019 = df_2019['Arthritis']

print(f"2019 Data loaded! Shape: {X_test_2019.shape}\n")

# ==========================================
# 2. DEFINE THE MODELS TO LOAD
# ==========================================
models_dir = os.path.join(base_dir, 'BRFSS 2021/Saved_Models_For_2019_Eval/')

model_files = {
    "GAN Imputation RUS Stacked Ensemble": "Stacked_Ensemble_RUS.pkl",
    "GAN Imputation ROS Stacked Ensemble": "GAN_ROS_Stacked_Ensemble.joblib",
    "GAN Imputation ROS XGBoost":          "GAN_ROS_XGBoost.joblib",
    "GAN Imputation ROS Logistic Reg":     "GAN_ROS_Logistic_Reg.joblib",
    "GAN Imputation RUS Logistic Reg":     "GAN_RUS_Logistic_Reg.joblib"
}

# ==========================================
# 3. EVALUATION FUNCTION
# ==========================================
def evaluate_model(model, X, y):
    """Calculates AUROC, Balanced Accuracy, Sensitivity, and Specificity"""
    # Get predictions and probabilities
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1] # Probabilities for the positive class (1)

    # Calculate Metrics
    auroc = roc_auc_score(y, y_proba)
    bal_acc = balanced_accuracy_score(y, y_pred)

    # Confusion Matrix for Sens/Spec
    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    # Composite Score
    composite_score = (auroc + bal_acc + sensitivity + specificity) / 4

    return {
        "Composite Score": composite_score,
        "AUROC": auroc,
        "Balanced Accuracy": bal_acc,
        "Overall Sensitivity": sensitivity,
        "Overall Specificity": specificity
    }

# ==========================================
# 4. RUN EVALUATION
# ==========================================
print("Starting evaluation on BRFSS 2019...\n")
results_list = []

for model_name, file_name in model_files.items():
    model_path = os.path.join(models_dir, file_name)

    try:
        print(f"Loading and evaluating: {model_name}...")

        # Load the model (joblib handles .pkl files perfectly)
        model = joblib.load(model_path)

        # Evaluate
        metrics = evaluate_model(model, X_test_2019, y_test_2019)

        # Add identifiers for the final table
        parts = model_name.split(" ")
        fill_method = " ".join(parts[:2]) # "GAN Imputation"
        sampling = parts[2]               # "RUS" or "ROS"
        base_model = " ".join(parts[3:])  # "Stacked Ensemble", etc.

        row = {
            "Fill method": fill_method,
            "Sampling": sampling,
            "Model": base_model,
            **metrics
        }
        results_list.append(row)

    except FileNotFoundError:
        print(f"  -> ERROR: Could not find '{file_name}' in {models_dir}. Please check the filename.")
    except Exception as e:
        print(f"  -> ERROR evaluating {model_name}: {e}")

# ==========================================
# 5. DISPLAY RESULTS
# ==========================================
print("\nEvaluation Complete! Here are the results for BRFSS 2019:\n")

# Convert to DataFrame for a clean, tabular view
results_df = pd.DataFrame(results_list)

# Sort by Composite Score descending
if not results_df.empty:
    results_df = results_df.sort_values(by="Composite Score", ascending=False).reset_index(drop=True)
    display(results_df) # Use print(results_df) if not in Jupyter/Colab

    # Save these results to a CSV with a new name
    results_out_path = os.path.join(base_dir, 'BRFSS_2019_10Percent_GAN_Evaluation_Results.csv')
    results_df.to_csv(results_out_path, index=False)
    print(f"\nResults saved to: {results_out_path}")

Loading BRFSS 2019 10% Sampled GAN-imputed test data...
2019 Data loaded! Shape: (41572, 62)

Starting evaluation on BRFSS 2019...

Loading and evaluating: GAN Imputation RUS Stacked Ensemble...
Loading and evaluating: GAN Imputation ROS Stacked Ensemble...
Loading and evaluating: GAN Imputation ROS XGBoost...
Loading and evaluating: GAN Imputation ROS Logistic Reg...
Loading and evaluating: GAN Imputation RUS Logistic Reg...

Evaluation Complete! Here are the results for BRFSS 2019:



,Fill method,Sampling,Model,Composite Score,AUROC,Balanced Accuracy,Overall Sensitivity,Overall Specificity
0,GAN Imputation,RUS,Stacked Ensemble,0.746600,0.806527,0.726624,0.801172,0.652076
1,GAN Imputation,ROS,Logistic Reg,0.741522,0.800781,0.721769,0.798242,0.645295
2,GAN Imputation,RUS,Logistic Reg,0.741074,0.800402,0.721298,0.805315,0.637280
3,GAN Imputation,ROS,XGBoost,0.736443,0.794424,0.717116,0.747089,0.687144
4,GAN Imputation,ROS,Stacked Ensemble,0.735792,0.793895,0.716425,0.737872,0.694977



Results saved to: /content/drive/MyDrive/Arthritis version 2/BRFSS_2019_10Percent_GAN_Evaluation_Results.csv
